# Phase 3: Model Training
## Credit Card Fraud Detection ML Pipeline

**Objective:** Train multiple baseline models with class weights to handle imbalance

**Models:**
1. Logistic Regression (baseline, interpretable)
2. Random Forest (tree-based, strong)
3. XGBoost (gradient boosting, state-of-the-art)

**Key:** All models use class_weight='balanced' to handle 0.17% fraud imbalance

**Outputs:**
- Trained models: `/models/model_lr.pkl`, `/models/model_rf.pkl`, `/models/model_xgb.pkl`
- Training report: `/reports/training_summary.txt`

## 1. Import Libraries & Load Data

In [1]:
import pandas as pd
import numpy as np
import pickle
from pathlib import Path
import time
import warnings
warnings.filterwarnings('ignore')

# ML models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

# Metrics
from sklearn.metrics import (
    precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report
)

# Set random seed
np.random.seed(42)

print('✓ Libraries imported successfully')

✓ Libraries imported successfully


## 2. Load Preprocessed Data

In [2]:
# Define paths
processed_path = Path('../data/processed')
models_path = Path('../models')
reports_path = Path('../reports')

# Load preprocessed data
X_train = pd.read_csv(processed_path / 'X_train_scaled.csv')
X_test = pd.read_csv(processed_path / 'X_test_scaled.csv')
y_train = pd.read_csv(processed_path / 'y_train.csv').iloc[:, 0]
y_test = pd.read_csv(processed_path / 'y_test.csv').iloc[:, 0]

print(f'✓ Data loaded successfully')
print(f'X_train shape: {X_train.shape}')
print(f'X_test shape: {X_test.shape}')
print(f'y_train shape: {y_train.shape}')
print(f'y_test shape: {y_test.shape}')

print(f'\nTrain set - Fraud ratio: {y_train.mean():.4f} ({y_train.mean()*100:.2f}%)')
print(f'Test set - Fraud ratio: {y_test.mean():.4f} ({y_test.mean()*100:.2f}%)')

✓ Data loaded successfully
X_train shape: (226980, 30)
X_test shape: (56746, 30)
y_train shape: (226980,)
y_test shape: (56746,)

Train set - Fraud ratio: 0.0017 (0.17%)
Test set - Fraud ratio: 0.0017 (0.17%)


## 3. Train Model 1: Logistic Regression (Baseline)

In [3]:
print('='*60)
print('MODEL 1: LOGISTIC REGRESSION (BASELINE)')
print('='*60)

# Initialize model with class weights
model_lr = LogisticRegression(
    class_weight='balanced',  # Handle imbalance
    max_iter=1000,
    random_state=42,
    n_jobs=-1
)

# Train
start_time = time.time()
model_lr.fit(X_train, y_train)
train_time_lr = time.time() - start_time

print(f'\n✓ Model trained in {train_time_lr:.2f} seconds')

# Predictions
y_train_pred_lr = model_lr.predict(X_train)
y_test_pred_lr = model_lr.predict(X_test)
y_test_proba_lr = model_lr.predict_proba(X_test)[:, 1]

# Metrics on test set
precision_lr = precision_score(y_test, y_test_pred_lr)
recall_lr = recall_score(y_test, y_test_pred_lr)
f1_lr = f1_score(y_test, y_test_pred_lr)
roc_auc_lr = roc_auc_score(y_test, y_test_proba_lr)

print(f'\nTest Set Metrics:')
print(f'  Precision: {precision_lr:.4f}')
print(f'  Recall: {recall_lr:.4f}')
print(f'  F1-score: {f1_lr:.4f}')
print(f'  ROC-AUC: {roc_auc_lr:.4f}')

# Train set metrics (check for overfitting)
y_train_proba_lr = model_lr.predict_proba(X_train)[:, 1]
train_roc_auc_lr = roc_auc_score(y_train, y_train_proba_lr)
print(f'\nTrain ROC-AUC: {train_roc_auc_lr:.4f}')
print(f'Overfitting indicator: {train_roc_auc_lr - roc_auc_lr:.4f}')

# Save model
model_path_lr = models_path / 'model_lr.pkl'
with open(model_path_lr, 'wb') as f:
    pickle.dump(model_lr, f)
print(f'\n✓ Model saved: {model_path_lr}')

MODEL 1: LOGISTIC REGRESSION (BASELINE)

✓ Model trained in 3.61 seconds

Test Set Metrics:
  Precision: 0.0296
  Recall: 0.8737
  F1-score: 0.0572
  ROC-AUC: 0.9610

Train ROC-AUC: 0.9801
Overfitting indicator: 0.0191

✓ Model saved: ..\models\model_lr.pkl


## 4. Train Model 2: Random Forest

In [4]:
print('='*60)
print('MODEL 2: RANDOM FOREST')
print('='*60)

# Initialize model with class weights
model_rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    min_samples_split=5,
    class_weight='balanced',  # Handle imbalance
    random_state=42,
    n_jobs=-1,
    verbose=0
)

# Train
start_time = time.time()
model_rf.fit(X_train, y_train)
train_time_rf = time.time() - start_time

print(f'\n✓ Model trained in {train_time_rf:.2f} seconds')

# Predictions
y_train_pred_rf = model_rf.predict(X_train)
y_test_pred_rf = model_rf.predict(X_test)
y_test_proba_rf = model_rf.predict_proba(X_test)[:, 1]

# Metrics on test set
precision_rf = precision_score(y_test, y_test_pred_rf)
recall_rf = recall_score(y_test, y_test_pred_rf)
f1_rf = f1_score(y_test, y_test_pred_rf)
roc_auc_rf = roc_auc_score(y_test, y_test_proba_rf)

print(f'\nTest Set Metrics:')
print(f'  Precision: {precision_rf:.4f}')
print(f'  Recall: {recall_rf:.4f}')
print(f'  F1-score: {f1_rf:.4f}')
print(f'  ROC-AUC: {roc_auc_rf:.4f}')

# Train set metrics (check for overfitting)
y_train_proba_rf = model_rf.predict_proba(X_train)[:, 1]
train_roc_auc_rf = roc_auc_score(y_train, y_train_proba_rf)
print(f'\nTrain ROC-AUC: {train_roc_auc_rf:.4f}')
print(f'Overfitting indicator: {train_roc_auc_rf - roc_auc_rf:.4f}')

# Save model
model_path_rf = models_path / 'model_rf.pkl'
with open(model_path_rf, 'wb') as f:
    pickle.dump(model_rf, f)
print(f'\n✓ Model saved: {model_path_rf}')

MODEL 2: RANDOM FOREST

✓ Model trained in 35.37 seconds

Test Set Metrics:
  Precision: 0.8933
  Recall: 0.7053
  F1-score: 0.7882
  ROC-AUC: 0.9544

Train ROC-AUC: 1.0000
Overfitting indicator: 0.0456

✓ Model saved: ..\models\model_rf.pkl


## 5. Train Model 3: XGBoost (Gradient Boosting)

In [6]:
print('='*60)
print('MODEL 3: XGBOOST (Gradient Boosting)')
print('='*60)

# Calculate scale_pos_weight for XGBoost (inverse of fraud ratio)
fraud_count = (y_train == 1).sum()
normal_count = (y_train == 0).sum()
scale_pos_weight = normal_count / fraud_count

print(f'\nScale pos weight (for XGBoost): {scale_pos_weight:.2f}')
print(f'(This penalizes fraud misclassification {scale_pos_weight:.0f}x more)')

# Initialize model with scale_pos_weight for imbalance
model_xgb = XGBClassifier(
    n_estimators=1000,
    max_depth=5,
    learning_rate=0.001,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,  # Handle imbalance
    random_state=42,
    n_jobs=-1,
    verbosity=0,
    eval_metric='logloss'
)

# Train
start_time = time.time()
model_xgb.fit(X_train, y_train, verbose=False)
train_time_xgb = time.time() - start_time

print(f'\n✓ Model trained in {train_time_xgb:.2f} seconds')

# Predictions
y_train_pred_xgb = model_xgb.predict(X_train)
y_test_pred_xgb = model_xgb.predict(X_test)
y_test_proba_xgb = model_xgb.predict_proba(X_test)[:, 1]

# Metrics on test set
precision_xgb = precision_score(y_test, y_test_pred_xgb)
recall_xgb = recall_score(y_test, y_test_pred_xgb)
f1_xgb = f1_score(y_test, y_test_pred_xgb)
roc_auc_xgb = roc_auc_score(y_test, y_test_proba_xgb)

print(f'\nTest Set Metrics:')
print(f'  Precision: {precision_xgb:.4f}')
print(f'  Recall: {recall_xgb:.4f}')
print(f'  F1-score: {f1_xgb:.4f}')
print(f'  ROC-AUC: {roc_auc_xgb:.4f}')

# Train set metrics (check for overfitting)
y_train_proba_xgb = model_xgb.predict_proba(X_train)[:, 1]
train_roc_auc_xgb = roc_auc_score(y_train, y_train_proba_xgb)
print(f'\nTrain ROC-AUC: {train_roc_auc_xgb:.4f}')
print(f'Overfitting indicator: {train_roc_auc_xgb - roc_auc_xgb:.4f}')

# Save model
model_path_xgb = models_path / 'model_xgb.pkl'
with open(model_path_xgb, 'wb') as f:
    pickle.dump(model_xgb, f)
print(f'\n✓ Model saved: {model_path_xgb}')

MODEL 3: XGBOOST (Gradient Boosting)

Scale pos weight (for XGBoost): 599.48
(This penalizes fraud misclassification 599x more)

✓ Model trained in 22.36 seconds

Test Set Metrics:
  Precision: 0.1941
  Recall: 0.8316
  F1-score: 0.3147
  ROC-AUC: 0.9611

Train ROC-AUC: 0.9973
Overfitting indicator: 0.0363

✓ Model saved: ..\models\model_xgb.pkl


## 6. Train Model 4: CatBoost

In [7]:
print('='*60)
print('MODEL 4: CATBOOST')
print('='*60)

# CatBoost is robust to categorical data and often requires less tuning
# Scale_pos_weight for imbalance handling
print(f'\nScale pos weight (for CatBoost): {scale_pos_weight:.2f}')

# Initialize model
model_cb = CatBoostClassifier(
    iterations=100,
    depth=6,
    learning_rate=0.1,
    subsample=0.8,
    scale_pos_weight=scale_pos_weight,  # Handle imbalance
    random_state=42,
    verbose=False,
    eval_metric='AUC'
)

# Train
start_time = time.time()
model_cb.fit(X_train, y_train, verbose=False)
train_time_cb = time.time() - start_time

print(f'\n✓ Model trained in {train_time_cb:.2f} seconds')

# Predictions
y_train_pred_cb = model_cb.predict(X_train)
y_test_pred_cb = model_cb.predict(X_test)
y_test_proba_cb = model_cb.predict_proba(X_test)[:, 1]

# Metrics on test set
precision_cb = precision_score(y_test, y_test_pred_cb)
recall_cb = recall_score(y_test, y_test_pred_cb)
f1_cb = f1_score(y_test, y_test_pred_cb)
roc_auc_cb = roc_auc_score(y_test, y_test_proba_cb)

print(f'\nTest Set Metrics:')
print(f'  Precision: {precision_cb:.4f}')
print(f'  Recall: {recall_cb:.4f}')
print(f'  F1-score: {f1_cb:.4f}')
print(f'  ROC-AUC: {roc_auc_cb:.4f}')

# Train set metrics (check for overfitting)
y_train_proba_cb = model_cb.predict_proba(X_train)[:, 1]
train_roc_auc_cb = roc_auc_score(y_train, y_train_proba_cb)
print(f'\nTrain ROC-AUC: {train_roc_auc_cb:.4f}')
print(f'Overfitting indicator: {train_roc_auc_cb - roc_auc_cb:.4f}')

# Save model
model_path_cb = models_path / 'model_cb.pkl'
with open(model_path_cb, 'wb') as f:
    pickle.dump(model_cb, f)
print(f'\n✓ Model saved: {model_path_cb}')

MODEL 4: CATBOOST

Scale pos weight (for CatBoost): 599.48

✓ Model trained in 6.26 seconds

Test Set Metrics:
  Precision: 0.5347
  Recall: 0.8105
  F1-score: 0.6444
  ROC-AUC: 0.9741

Train ROC-AUC: 0.9999
Overfitting indicator: 0.0258

✓ Model saved: ..\models\model_cb.pkl


## 6. Model Comparison (Baseline)

In [8]:
print('='*60)
print('BASELINE MODEL COMPARISON')
print('='*60)

# Create comparison table
comparison_df = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest', 'XGBoost', 'CatBoost'],
    'Precision': [precision_lr, precision_rf, precision_xgb, precision_cb],
    'Recall': [recall_lr, recall_rf, recall_xgb, recall_cb],
    'F1-Score': [f1_lr, f1_rf, f1_xgb, f1_cb],
    'ROC-AUC': [roc_auc_lr, roc_auc_rf, roc_auc_xgb, roc_auc_cb],
    'Train Time (s)': [train_time_lr, train_time_rf, train_time_xgb, train_time_cb]
})

# Sort by ROC-AUC
comparison_df = comparison_df.sort_values('ROC-AUC', ascending=False).reset_index(drop=True)

print(f'\n{comparison_df.to_string(index=False)}')

# Save comparison
comparison_df.to_csv(reports_path / 'baseline_model_comparison.csv', index=False)
print(f'\n✓ Comparison saved: baseline_model_comparison.csv')

# Identify best model by ROC-AUC
best_idx = 0
best_model_name = comparison_df.loc[best_idx, 'Model']
best_roc_auc = comparison_df.loc[best_idx, 'ROC-AUC']

print(f'\n⭐ Best baseline model: {best_model_name} (ROC-AUC: {best_roc_auc:.4f})')

BASELINE MODEL COMPARISON

              Model  Precision   Recall  F1-Score  ROC-AUC  Train Time (s)
           CatBoost   0.534722 0.810526  0.644351 0.974106        6.258852
            XGBoost   0.194103 0.831579  0.314741 0.961061       22.356417
Logistic Regression   0.029558 0.873684  0.057182 0.961020        3.611747
      Random Forest   0.893333 0.705263  0.788235 0.954373       35.370847

✓ Comparison saved: baseline_model_comparison.csv

⭐ Best baseline model: CatBoost (ROC-AUC: 0.9741)


## 7. Detailed Analysis: Confusion Matrices

In [9]:
print('='*60)
print('CONFUSION MATRIX ANALYSIS')
print('='*60)

models = {
    'Logistic Regression': y_test_pred_lr,
    'Random Forest': y_test_pred_rf,
    'XGBoost': y_test_pred_xgb,
    'CatBoost': y_test_pred_cb
}

for model_name, y_pred in models.items():
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    
    print(f'\n{model_name}:')
    print(f'  True Negatives (TN): {tn:,}')
    print(f'  False Positives (FP): {fp:,}')
    print(f'  False Negatives (FN): {fn:,}')
    print(f'  True Positives (TP): {tp:,}')
    print(f'  Caught fraud: {tp} / {tp + fn} = {tp/(tp+fn)*100:.1f}%')

CONFUSION MATRIX ANALYSIS

Logistic Regression:
  True Negatives (TN): 53,926
  False Positives (FP): 2,725
  False Negatives (FN): 12
  True Positives (TP): 83
  Caught fraud: 83 / 95 = 87.4%

Random Forest:
  True Negatives (TN): 56,643
  False Positives (FP): 8
  False Negatives (FN): 28
  True Positives (TP): 67
  Caught fraud: 67 / 95 = 70.5%

XGBoost:
  True Negatives (TN): 56,323
  False Positives (FP): 328
  False Negatives (FN): 16
  True Positives (TP): 79
  Caught fraud: 79 / 95 = 83.2%

CatBoost:
  True Negatives (TN): 56,584
  False Positives (FP): 67
  False Negatives (FN): 18
  True Positives (TP): 77
  Caught fraud: 77 / 95 = 81.1%


## 8. Training Summary Report

In [10]:
training_report = f"""
================================================================================
MODEL TRAINING SUMMARY REPORT
================================================================================

1. TRAINING DATA
   - Training set size: {X_train.shape[0]:,} samples
   - Test set size: {X_test.shape[0]:,} samples
   - Features: {X_train.shape[1]}
   - Fraud ratio (train): {y_train.mean():.4f} ({y_train.sum():,} fraud cases)
   - Fraud ratio (test): {y_test.mean():.4f} ({y_test.sum():,} fraud cases)

2. IMBALANCE HANDLING STRATEGY
   - Method: Class weights / Scale Pos Weight (built-in to models)
   - Logistic Regression: class_weight='balanced'
   - Random Forest: class_weight='balanced'
   - XGBoost: scale_pos_weight={scale_pos_weight:.2f}
   - CatBoost: scale_pos_weight={scale_pos_weight:.2f}

3. MODEL 1: LOGISTIC REGRESSION (BASELINE)
   Training time: {train_time_lr:.2f} seconds
   
   Test Set Metrics:
   - Precision: {precision_lr:.4f}
   - Recall: {recall_lr:.4f}
   - F1-Score: {f1_lr:.4f}
   - ROC-AUC: {roc_auc_lr:.4f}
   
   Overfitting Analysis:
   - Train ROC-AUC: {train_roc_auc_lr:.4f}
   - Test ROC-AUC: {roc_auc_lr:.4f}
   - Gap: {train_roc_auc_lr - roc_auc_lr:.4f} (acceptable)

4. MODEL 2: RANDOM FOREST
   Training time: {train_time_rf:.2f} seconds
   Parameters: n_estimators=100, max_depth=15, min_samples_split=5
   
   Test Set Metrics:
   - Precision: {precision_rf:.4f}
   - Recall: {recall_rf:.4f}
   - F1-Score: {f1_rf:.4f}
   - ROC-AUC: {roc_auc_rf:.4f}
   
   Overfitting Analysis:
   - Train ROC-AUC: {train_roc_auc_rf:.4f}
   - Test ROC-AUC: {roc_auc_rf:.4f}
   - Gap: {train_roc_auc_rf - roc_auc_rf:.4f} (acceptable)

5. MODEL 3: XGBOOST
   Training time: {train_time_xgb:.2f} seconds
   Parameters: n_estimators=100, max_depth=5, learning_rate=0.1
   
   Test Set Metrics:
   - Precision: {precision_xgb:.4f}
   - Recall: {recall_xgb:.4f}
   - F1-Score: {f1_xgb:.4f}
   - ROC-AUC: {roc_auc_xgb:.4f}
   
   Overfitting Analysis:
   - Train ROC-AUC: {train_roc_auc_xgb:.4f}
   - Test ROC-AUC: {roc_auc_xgb:.4f}
   - Gap: {train_roc_auc_xgb - roc_auc_xgb:.4f} (acceptable)

6. MODEL 4: CATBOOST
   Training time: {train_time_cb:.2f} seconds
   Parameters: iterations=100, depth=6, learning_rate=0.1
   
   Test Set Metrics:
   - Precision: {precision_cb:.4f}
   - Recall: {recall_cb:.4f}
   - F1-Score: {f1_cb:.4f}
   - ROC-AUC: {roc_auc_cb:.4f}
   
   Overfitting Analysis:
   - Train ROC-AUC: {train_roc_auc_cb:.4f}
   - Test ROC-AUC: {roc_auc_cb:.4f}
   - Gap: {train_roc_auc_cb - roc_auc_cb:.4f} (acceptable)

7. BASELINE COMPARISON (Sorted by ROC-AUC)
   Best Model: {best_model_name}
   ROC-AUC Score: {best_roc_auc:.4f}
   
   Model Rankings:
{pd.concat([pd.Series(['1. ', '2. ', '3. ', '4. ']), comparison_df['Model'], pd.Series([f"({roc_auc:.4f})" for roc_auc in comparison_df['ROC-AUC']])], axis=1).to_string(header=False, index=False)}

8. SAVED ARTIFACTS
   Models:
   - ../models/model_lr.pkl
   - ../models/model_rf.pkl
   - ../models/model_xgb.pkl
   - ../models/model_cb.pkl
   
   Comparison:
   - ../reports/baseline_model_comparison.csv

9. NEXT STEPS
   ✓ Phase 4: Hyperparameter Tuning (optimize {best_model_name})
   ✓ Phase 5: Evaluation (comprehensive metrics on test set)
   ✓ Phase 6: Model Selection (final model comparison)
   ✓ Phase 7: SHAP Explainability

================================================================================
"""

print(training_report)

# Save report
with open(reports_path / 'training_summary.txt', 'w') as f:
    f.write(training_report)

print('✓ Saved: training_summary.txt')


MODEL TRAINING SUMMARY REPORT

1. TRAINING DATA
   - Training set size: 226,980 samples
   - Test set size: 56,746 samples
   - Features: 30
   - Fraud ratio (train): 0.0017 (378 fraud cases)
   - Fraud ratio (test): 0.0017 (95 fraud cases)

2. IMBALANCE HANDLING STRATEGY
   - Method: Class weights / Scale Pos Weight (built-in to models)
   - Logistic Regression: class_weight='balanced'
   - Random Forest: class_weight='balanced'
   - XGBoost: scale_pos_weight=599.48
   - CatBoost: scale_pos_weight=599.48

3. MODEL 1: LOGISTIC REGRESSION (BASELINE)
   Training time: 3.61 seconds
   
   Test Set Metrics:
   - Precision: 0.0296
   - Recall: 0.8737
   - F1-Score: 0.0572
   - ROC-AUC: 0.9610
   
   Overfitting Analysis:
   - Train ROC-AUC: 0.9801
   - Test ROC-AUC: 0.9610
   - Gap: 0.0191 (acceptable)

4. MODEL 2: RANDOM FOREST
   Training time: 35.37 seconds
   Parameters: n_estimators=100, max_depth=15, min_samples_split=5
   
   Test Set Metrics:
   - Precision: 0.8933
   - Recall: 0.7

##  Model Training Complete

**Summary:**
- ✓ Trained 4 baseline models with class weights
- ✓ All models evaluated on test set
- ✓ No significant overfitting detected
- ✓ Best model identified by ROC-AUC score

**Models trained:**
1. Logistic Regression (baseline)
2. Random Forest (tree-based)
3. XGBoost (gradient boosting)
4. CatBoost (advanced gradient boosting)

**Next:** Run `04_tuning.ipynb` to optimize the best model